# Tablas Comparativas de Métricas de Evaluación XAI

Genera 3 tablas comparativas (una por corte @k) con columnas:
- **AggDiv** — Diversidad Agregada (nivel usuario)
- **IXD** — Inter-eXplanation Diversity (nivel usuario)
- **MIL** — Mean Inter-List Diversity (nivel sistema)
- **ECS** — Explanation Consistency Score (media sobre `hotel_recomendado`)

Cada fila es un algoritmo. El valor de ECS en la tabla es la **media de ECS** sobre todos los hoteles recomendados con ≥2 usuarios.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from IPython.display import display, HTML

## 1. Configuración de rutas

In [ ]:
MODO = 'semi'   # opciones: 'muestra' | 'semi' | 'completo'

BASE_DIR = os.path.join('..', '..', 'output', f'metricas_evaluacion_{MODO}')

if not os.path.exists(BASE_DIR):
    print(f'⚠️  Directorio no encontrado: {BASE_DIR}')
else:
    csvs = glob.glob(os.path.join(BASE_DIR, '*.csv'))
    print(f'✅ Directorio encontrado. CSVs disponibles: {len(csvs)}')
    for f in sorted(csvs):
        print(' -', os.path.basename(f))

## 2. Carga y agregación de datos

Se detecta automáticamente qué métrica contiene cada CSV por su nombre de fichero.

In [ ]:
METRICAS_CONOCIDAS = ['AggDiv', 'IXD', 'MIL', 'ECS']


def detectar_metrica(filename):
    basename = os.path.basename(filename)
    for m in METRICAS_CONOCIDAS:
        if f'_{m}_' in basename or f'_{m}.' in basename:
            return m
    return None


def cargar_todos_los_csvs(base_dir):
    """
    Lee todos los CSVs y devuelve un DataFrame unificado con columnas:
    algoritmo | metrica | valor | valor@1 | valor@3 | valor@5

    Para ECS (granularidad hotel): agrega por media sobre hotel_recomendado
    antes de añadir la fila, para que quede 1 valor por algoritmo (igual que MIL).
    """
    registros = []
    csvs = glob.glob(os.path.join(base_dir, '*.csv'))

    for path in sorted(csvs):
        metrica = detectar_metrica(path)
        if metrica is None:
            print(f'⚠️  No se detectó métrica en: {os.path.basename(path)}, ignorando.')
            continue

        df = pd.read_csv(path)

        col_base = metrica
        col_1    = f'{metrica}@1'
        col_3    = f'{metrica}@3'
        col_5    = f'{metrica}@5'

        if metrica == 'ECS':
            # ECS tiene una fila por hotel_recomendado → agregar por media
            algoritmo = df['algoritmo'].iloc[0] if 'algoritmo' in df.columns else None
            if algoritmo is None:
                continue
            registros.append({
                'algoritmo': algoritmo,
                'metrica':   metrica,
                'valor':     df[col_base].dropna().mean() if col_base in df.columns else np.nan,
                'valor@1':   df[col_1].dropna().mean()    if col_1    in df.columns else np.nan,
                'valor@3':   df[col_3].dropna().mean()    if col_3    in df.columns else np.nan,
                'valor@5':   df[col_5].dropna().mean()    if col_5    in df.columns else np.nan,
                'n_hoteles': len(df),
            })
        else:
            # AggDiv, IXD (fila por usuario) y MIL (fila única)
            for _, row in df.iterrows():
                algoritmo = row.get('algoritmo', None)
                if algoritmo is None:
                    continue
                registros.append({
                    'algoritmo': algoritmo,
                    'metrica':   metrica,
                    'valor':     row.get(col_base, np.nan),
                    'valor@1':   row.get(col_1,    np.nan),
                    'valor@3':   row.get(col_3,    np.nan),
                    'valor@5':   row.get(col_5,    np.nan),
                    'n_hoteles': np.nan,
                })

    return pd.DataFrame(registros)


df_raw = cargar_todos_los_csvs(BASE_DIR)
print(f'Registros cargados: {len(df_raw)}')
print(f'Métricas presentes: {df_raw["metrica"].unique().tolist()}')
display(df_raw.head(12))

## 3. Construcción de las tablas por @k

In [ ]:
def construir_tabla_k(df_raw, k):
    """
    Construye un DataFrame con filas=algoritmos y columnas=métricas
    para el corte @k (1, 3 o 5).

    Columnas: AggDiv | IXD | MIL | ECS
    - AggDiv, IXD: media sobre usuarios
    - MIL:         valor único del sistema
    - ECS:         media sobre hotel_recomendado (ya agregada en cargar_todos_los_csvs)
    """
    columna_k = f'valor@{k}' if k is not None else 'valor'
    algoritmos = sorted(df_raw['algoritmo'].unique())
    filas = []

    for alg in algoritmos:
        fila = {'Algoritmo': alg}
        for m in METRICAS_CONOCIDAS:
            subset = df_raw[(df_raw['algoritmo'] == alg) & (df_raw['metrica'] == m)]
            if not subset.empty:
                fila[m] = round(subset[columna_k].mean(), 6)
            else:
                fila[m] = np.nan
        filas.append(fila)

    tabla = pd.DataFrame(filas).set_index('Algoritmo')
    # Reordenar columnas
    cols_presentes = [m for m in METRICAS_CONOCIDAS if m in tabla.columns]
    return tabla[cols_presentes]


tabla_global = construir_tabla_k(df_raw, k=None)
tabla_1      = construir_tabla_k(df_raw, k=1)
tabla_3      = construir_tabla_k(df_raw, k=3)
tabla_5      = construir_tabla_k(df_raw, k=5)

print(f'Algoritmos detectados: {list(tabla_1.index)}')
print(f'Columnas: {list(tabla_1.columns)}')

## 4. Visualización de las tablas

> **Nota sobre ECS**: el valor mostrado es la **media de ECS sobre todos los `hotel_recomendado`** con ≥2 usuarios para ese algoritmo. Un valor bajo indica que los explicadores son heterogéneos entre usuarios (alta personalización); un valor alto indica que todos los usuarios reciben los mismos explicadores para ese hotel (alta consistencia).

In [ ]:
# Dirección de la escala por métrica:
#   AggDiv → más alto es más diverso (verde oscuro = mejor)
#   IXD    → más alto es más diverso (verde oscuro = mejor)
#   MIL    → más alto es más personalizado (verde oscuro = mejor)
#   ECS    → más alto es más consistente / menos personalizado
#             → invertimos la escala (verde oscuro = más consistente)

ESCALAS = {
    'AggDiv': {'cmap': 'YlGn',  'desc': '↑ más diverso'},
    'IXD':    {'cmap': 'YlGn',  'desc': '↑ más diverso'},
    'MIL':    {'cmap': 'YlGn',  'desc': '↑ más personalizado'},
    'ECS':    {'cmap': 'YlOrRd','desc': '↑ más consistente (menos personalizado)'},
}


def mostrar_tabla(tabla, titulo):
    """Muestra una tabla con degradado de color por columna."""
    styler = tabla.style.set_caption(titulo).format('{:.4f}', na_rep='N/A')

    for col in tabla.columns:
        cmap = ESCALAS.get(col, {}).get('cmap', 'YlGn')
        styler = styler.background_gradient(cmap=cmap, subset=[col], axis=0)

    styler = styler.set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '15px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding-bottom', '6px')]},
        {'selector': 'th',
         'props': [('background-color', '#2c3e50'), ('color', 'white'),
                   ('text-align', 'center'), ('padding', '8px 14px')]},
        {'selector': 'td',
         'props': [('text-align', 'center'), ('padding', '6px 14px')]},
    ])
    display(styler)

    # Leyenda de columnas
    leyenda = ' | '.join(
        f'<b>{col}</b>: {ESCALAS.get(col, {}).get("desc", "")}'
        for col in tabla.columns if col in ESCALAS
    )
    display(HTML(f'<p style="font-size:11px;color:#555;margin-top:2px">{leyenda}</p>'))
    print()

In [ ]:
mostrar_tabla(tabla_global, '📊 Métricas de Evaluación — global (todos los explicadores)')
mostrar_tabla(tabla_1,      '📊 Métricas de Evaluación @ 1')
mostrar_tabla(tabla_3,      '📊 Métricas de Evaluación @ 3')
mostrar_tabla(tabla_5,      '📊 Métricas de Evaluación @ 5')

## 5. Detalle de ECS por algoritmo y hotel_recomendado

Muestra los hoteles con mayor y menor consistencia para cada algoritmo.

In [ ]:
N_TOP = 10   # hoteles a mostrar por extremo

csvs_ecs = sorted(glob.glob(os.path.join(BASE_DIR, '*_ECS_*.csv')))

for path in csvs_ecs:
    df = pd.read_csv(path)
    if df.empty or 'ECS' not in df.columns:
        continue
    algoritmo = df['algoritmo'].iloc[0] if 'algoritmo' in df.columns else os.path.basename(path)
    df_sorted = df.sort_values('ECS', ascending=False)

    print(f'\n══════════════════════════════════════════════')
    print(f'  {algoritmo}  —  {len(df)} hoteles recomendados')
    print(f'  ECS medio: {df["ECS"].mean():.4f}   mediana: {df["ECS"].median():.4f}')
    print(f'══════════════════════════════════════════════')

    cols_mostrar = ['hotel_recomendado', 'n_usuarios', 'ECS', 'ECS@1', 'ECS@3', 'ECS@5']
    cols_ok = [c for c in cols_mostrar if c in df.columns]

    print(f'\n  Top {N_TOP} hoteles con mayor consistencia (ECS alto):')
    display(df_sorted.head(N_TOP)[cols_ok].reset_index(drop=True))

    print(f'\n  Top {N_TOP} hoteles con menor consistencia (ECS bajo):')
    display(df_sorted.tail(N_TOP)[cols_ok].reset_index(drop=True))

## 6. Exportar tablas a CSV

In [ ]:
OUTPUT_DIR = os.path.join('..', '..', 'output', 'visualizacion_tablas')
os.makedirs(OUTPUT_DIR, exist_ok=True)

tabla_global.to_csv(os.path.join(OUTPUT_DIR, 'tabla_metricas_global.csv'))
tabla_1.to_csv(os.path.join(OUTPUT_DIR, 'tabla_metricas_at1.csv'))
tabla_3.to_csv(os.path.join(OUTPUT_DIR, 'tabla_metricas_at3.csv'))
tabla_5.to_csv(os.path.join(OUTPUT_DIR, 'tabla_metricas_at5.csv'))

print(f'✅ Tablas exportadas a: {OUTPUT_DIR}')
for f in ['tabla_metricas_global.csv', 'tabla_metricas_at1.csv',
          'tabla_metricas_at3.csv', 'tabla_metricas_at5.csv']:
    print(f'   - {f}')